# Ch 6 — 5-state Random Walk: TD(0) vs MC

**复现 Sutton & Barto 图 6.2** + 数值 trace。

## Setup
- 状态：A=0, B=1, C=2, D=3, E=4（中间起点 C=2）
- 转移：每步左右等概；左终止得 0，右终止得 1
- **真实 V**: [1/6, 2/6, 3/6, 4/6, 5/6]

## 学什么
1. MC 跟 TD(0) 估 V 的**收敛速度差异**
2. 不同 α 下的 trade-off
3. TD bootstrap 为什么 sample efficient

## 链回
- [[ch06-temporal-difference-learning]]
- [[_anki/ch06-td-cards]]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

np.random.seed(42)

# Environment
N_STATES = 5  # A B C D E
START = 2  # C
TRUE_V = np.array([1/6, 2/6, 3/6, 4/6, 5/6])

def run_episode():
    """Return list of (state, reward) trajectory ending at terminal."""
    s = START
    traj = []
    while True:
        a = np.random.choice([-1, 1])  # left or right
        s_next = s + a
        if s_next == -1:
            traj.append((s, 0))  # left terminal, reward 0
            return traj, 0
        elif s_next == N_STATES:
            traj.append((s, 1))  # right terminal, reward 1
            return traj, 1
        else:
            traj.append((s, 0))
            s = s_next

# Test
traj, G = run_episode()
print(f'Sample episode (length {len(traj)}): {traj}, return={G}')

## TD(0) 实现

**Update**: V(s) ← V(s) + α · [r + γV(s') - V(s)]

我们用 γ=1（episodic），所以 target = r + V(s') 当 s' 非终止，否则 target = r。

In [ ]:
def td0(n_episodes, alpha):
    V = np.ones(N_STATES) * 0.5  # init to 0.5
    rmse_history = []
    for ep in range(n_episodes):
        s = START
        while True:
            a = np.random.choice([-1, 1])
            s_next = s + a
            if s_next == -1:
                target = 0  # terminal, no bootstrap
                V[s] += alpha * (target - V[s])
                break
            elif s_next == N_STATES:
                target = 1
                V[s] += alpha * (target - V[s])
                break
            else:
                target = 0 + V[s_next]  # bootstrap
                V[s] += alpha * (target - V[s])
                s = s_next
        rmse = np.sqrt(np.mean((V - TRUE_V) ** 2))
        rmse_history.append(rmse)
    return V, rmse_history

V_td, rmse_td = td0(n_episodes=100, alpha=0.1)
print(f'TD(0) final V: {np.round(V_td, 3)}')
print(f'True V      : {np.round(TRUE_V, 3)}')
print(f'Final RMSE  : {rmse_td[-1]:.4f}')

## MC 实现

**Update**: V(s) ← V(s) + α · [G - V(s)]，G 是 episode return（这里就是 0 或 1）。

用 every-visit MC。

In [ ]:
def mc(n_episodes, alpha):
    V = np.ones(N_STATES) * 0.5
    rmse_history = []
    for ep in range(n_episodes):
        traj, G = run_episode()
        for s, _ in traj:
            V[s] += alpha * (G - V[s])
        rmse = np.sqrt(np.mean((V - TRUE_V) ** 2))
        rmse_history.append(rmse)
    return V, rmse_history

V_mc, rmse_mc = mc(n_episodes=100, alpha=0.05)
print(f'MC final V: {np.round(V_mc, 3)}')
print(f'True V    : {np.round(TRUE_V, 3)}')
print(f'Final RMSE: {rmse_mc[-1]:.4f}')

## 数值 trace：前 10 个 episode 各 state 的 V 演化

In [ ]:
def td0_trace(n_episodes, alpha):
    V = np.ones(N_STATES) * 0.5
    history = [V.copy()]
    for ep in range(n_episodes):
        s = START
        while True:
            a = np.random.choice([-1, 1])
            s_next = s + a
            if s_next == -1:
                V[s] += alpha * (0 - V[s])
                break
            elif s_next == N_STATES:
                V[s] += alpha * (1 - V[s])
                break
            else:
                V[s] += alpha * (V[s_next] - V[s])
                s = s_next
        history.append(V.copy())
    return history

np.random.seed(7)
hist = td0_trace(n_episodes=10, alpha=0.2)
print(f'{"Episode":<10}' + ' '.join(f'V({c})' + ' '*4 for c in 'ABCDE'))
for i, V in enumerate(hist):
    print(f'{i:<10}' + ' '.join(f'{v:.3f}  ' for v in V))

## 多 α 的收敛比较（复现 Sutton 图 6.2 right panel）

TD(0) 跟 MC 在多 α 下平均 RMSE 曲线——TD(0) 显著更 sample efficient。

In [ ]:
def avg_rmse(method, n_runs, n_episodes, alpha):
    rmses = np.zeros(n_episodes)
    for _ in range(n_runs):
        np.random.seed(None)
        _, h = method(n_episodes, alpha)
        rmses += np.array(h)
    return rmses / n_runs

fig, ax = plt.subplots(figsize=(8, 5))
for alpha in [0.05, 0.1, 0.15]:
    r = avg_rmse(td0, 50, 100, alpha)
    ax.plot(r, label=f'TD α={alpha}', linestyle='-')
for alpha in [0.01, 0.02, 0.04]:
    r = avg_rmse(mc, 50, 100, alpha)
    ax.plot(r, label=f'MC α={alpha}', linestyle='--')
ax.set_xlabel('Episode')
ax.set_ylabel('RMSE (averaged over 50 runs)')
ax.set_title('TD(0) vs MC on 5-state Random Walk')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 关键观察

1. **TD(0) 收敛更快**——bootstrap 让每个 sample 立刻 update（不等 episode 终止）
2. **TD(0) 对 α 更宽容**——MC variance 大，α 必须更小
3. **TD asymptotic bias**——TD 在 α 大时可能有 systematic bias（但在 random walk 上不明显）

## Interview 启示

- 手撕 TD 跟 MC 的 trade-off：TD bias 高/variance 低/online；MC bias 0/variance 高/episodic
- 实战 deep RL（DQN/PPO）几乎都 bootstrap——MC return 在 long horizon 上 variance 太炸
- 但 advantage estimation 一般用 GAE（TD 跟 MC 之间 λ-插值）

## 链回
- [[ch06-temporal-difference-learning]] §3.1 TD(0)
- [[ch12-gae-step-by-step.ipynb]] — GAE 的 λ-插值具体看下一本 NB